# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [ ]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 💾 Chargement des données
Les données `.dat` sont chargées nativement par Pyomo via `model.create_instance(...)` dans la section du modèle.

## 🔹 Model

In [ ]:
from pyomo.environ import *

model = AbstractModel()

## 🔹 Sets

In [ ]:
model.MATIERES = Set()
model.PRODUITS = Set()
model.ARC = Set(dimen=2, initialize=lambda m: [(i,j) for i in m.MATIERES for j in m.PRODUITS])

## 🔹 Parameters

In [ ]:
model.dispo = Param(model.MATIERES, within=NonNegativeReals)
model.gain = Param(model.PRODUITS, within=NonNegativeReals)
model.engagement = Param(model.PRODUITS, within=NonNegativeReals)
model.utilisation = Param(model.MATIERES, model.PRODUITS, within=NonNegativeReals)

## 🔹 Variables

In [ ]:
model.X = Var(model.PRODUITS, domain=NonNegativeReals)

## 🔹 Data

In [ ]:
model = model.create_instance('../data/forets_data.dat')

## 🔹 Constraints

In [ ]:
model.c_for_0 = ConstraintList()
for p in model.PRODUITS:
    model.c_for_0.add(model.X[p] >= model.engagement[p])
model.c_for_1 = ConstraintList()
for m in model.MATIERES:
    model.c_for_1.add(sum(model.utilisation[m,p] * model.X[p] for p in model.PRODUITS) <= model.dispo[m])

## 🔹 Objective

In [ ]:
model.obj = Objective(expr=sum(model.gain[p] * model.X[p] for p in model.PRODUITS), sense=maximize)

## ⚙️ Résolution du modèle

In [ ]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

## 🎯 Valeur de la fonction objective

In [ ]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

## 📊 Valeurs optimales des variables

In [ ]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')